# Per-layer Pfaffian profile of GPT-2, BERT, ResNet-50, ViT

Reproducer for `bench/architectures/`. Run top-to-bottom in a fresh Colab or local environment.

**Setup**
```
pip install eml-cost eml-cost-torch torch transformers torchvision matplotlib
```

Output: per-layer Pfaffian profile JSON + matplotlib visualization for each architecture.

Note: models are instantiated with default config (random weights). We do not need to run forward; we only need the architecture tree.

In [ ]:
from eml_cost_torch import profile_dict
from collections import Counter
import json

## 1. GPT-2 (124M)

In [ ]:
from transformers import GPT2Config, GPT2Model
model = GPT2Model(GPT2Config())
rows = profile_dict(model)
print(f'GPT-2: {len(rows)} layers')
print(f'Coverage: {sum(1 for r in rows if not r["is_unknown"])}/{len(rows)}')
print(f'Top axes:')
for ax, n in Counter(r["axes"] for r in rows if not r["is_unknown"]).most_common(5):
    print(f'  {ax}: {n}')
print(f'Pfaffian-not-EML layers: {sum(1 for r in rows if r["is_pfaffian_not_eml"])}')

## 2. BERT-base (110M)

In [ ]:
from transformers import BertConfig, BertModel
model = BertModel(BertConfig())
rows = profile_dict(model)
print(f'BERT: {len(rows)} layers')
print(f'Coverage: {sum(1 for r in rows if not r["is_unknown"])}/{len(rows)}')
print(f'Top axes:')
for ax, n in Counter(r["axes"] for r in rows if not r["is_unknown"]).most_common(5):
    print(f'  {ax}: {n}')

## 3. ResNet-50 (25M)

In [ ]:
from torchvision.models import resnet50
model = resnet50(weights=None)
rows = profile_dict(model)
print(f'ResNet-50: {len(rows)} layers')
print(f'Coverage: {sum(1 for r in rows if not r["is_unknown"])}/{len(rows)}')
print(f'Top axes:')
for ax, n in Counter(r["axes"] for r in rows if not r["is_unknown"]).most_common(5):
    print(f'  {ax}: {n}')

## 4. ViT-B/16 (86M)

In [ ]:
from torchvision.models import vit_b_16
model = vit_b_16(weights=None)
rows = profile_dict(model)
print(f'ViT-B/16: {len(rows)} layers')
print(f'Coverage: {sum(1 for r in rows if not r["is_unknown"])}/{len(rows)}')
print(f'Top axes:')
for ax, n in Counter(r["axes"] for r in rows if not r["is_unknown"]).most_common(5):
    print(f'  {ax}: {n}')
print(f'Pfaffian-not-EML layers (GELU/erf): {sum(1 for r in rows if r["is_pfaffian_not_eml"])}')

## What the profile tells you

**Layer-level vs block-level profile.** This profile measures each leaf module in isolation. A `nn.Linear` layer has Pfaffian-r 0 because the matmul is polynomial in its inputs. A `nn.GELU` layer has r=3 because it contains erf. A LayerNorm has r=1.

When you compose them mathematically — write out the full block expression — you get a higher r. From cross-modal v3 substrate run, a full Transformer block (residual + LayerNorm + multi-head attention + FFN) lands at `p7-d8-w3-c0` (r=7, depth=11). That's the *blockwise* claim; this notebook shows the *componentwise* table.

**The headline finding from the v3 corpus is the componentwise vs blockwise distinction.** Each component is in a low-r class; the composed block is in a high-r class. No single component lands in the same class as the full block.

**ViT has 12 Pfaffian-not-EML layers.** Each GELU contains erf, which is Pfaffian but lives outside the elementary class. The bell ping in [Sound of Math](https://1op.io/learn/sound-lane5) marks where these step outside.

**Coverage gaps are honest.** GPT-2 uses HuggingFace's custom `Conv1D` (not `nn.Linear`) which isn't in the registry — that's why coverage is ~50%. BERT uses `nn.Linear` so coverage is ~92%. ResNet and ViT use stock torchvision modules so coverage is 100%.